In [1]:
import temporaldata as td
import experanto as exp
from interface_experanto_temporaldata import Interface, screen_fn

import numpy as np
import h5py

# Temporaldata -> Experanto -> Temporaldata

In [2]:
def td_check_equality(first: [td.Data, td.RegularTimeSeries, td.IrregularTimeSeries, td.Interval], second: [td.Data, td.RegularTimeSeries, td.IrregularTimeSeries, td.Interval]) -> bool:
    for key in first.keys():
        if key in second.keys():
            if isinstance(getattr(first, key), np.ndarray):
                if (getattr(first, key) != getattr(second, key)).all():
                    print(f"Value mismatch for key '{key}': {getattr(first, key)} != {getattr(second, key)}")
                    return False
            elif isinstance(getattr(first, key), (int, float, str, bool, list, dict, np.bool_)):
                if getattr(first, key) != getattr(second, key):
                    print(f"Value mismatch for key '{key}': {getattr(first, key)} != {getattr(second, key)}")
                    return False
            else:
                if not td_check_equality(getattr(first, key), getattr(second, key)):
                    return False
        else:
            print(f"Key '{key}' not found in second object.")
            return False
    return True
            

In [3]:
path_to_processed_data = "./Temporaldata/data/processed/perich_miller_population_2018/c_20131003_center_out_reaching.h5"
with h5py.File(path_to_processed_data, "r") as f:
    td_dataset = td.Data.from_hdf5(f, lazy=False)  
print("Dataset: ", td_dataset)


Dataset:  Data(
brainset=Data(
brainsets_version='0.2.0',
derived_version='1.0.0',
description='This dataset contains electrophysiology and behavioral data from three macaques performing either a center-out task or a continuous random target acquisition task. Neural activity was recorded from chronically-implanted electrode arrays in the primary motor cortex (M1) or dorsal premotor cortex (PMd) of four rhesus macaque monkeys. A subset of sessions includes recordings from both regions simultaneously. The data contains spiking activity—manually spike sorted in three subjects, and threshold crossings in the fourth subject—obtained from up to 192 electrodes per session, cursor position and velocity, and other task related metadata.',
id='perich_miller_population_2018',
origin_version='dandi/000688/draft',
source='https://dandiarchive.org/dandiset/000688',
temporaldata_version='0.1.1',
_absolute_start=0.0,
),
cursor=IrregularTimeSeries(
  timestamps=[66321],
  acc=[66321, 2],
  pos=[66321, 

In [4]:
interface_dict = Interface({"example": td_dataset})
interface_example = interface_dict.process_as_experanto("example")
print("interface_example: ", interface_example)


Td -> Exp
interface_example:  <interface_experanto_temporaldata.InterfaceExperiment.InterfaceExperiment object at 0x1457f0450>


In [5]:
interface_dict_backward = Interface({"example": interface_example})
interface_example_backward = interface_dict_backward.process_as_temporaldata("example")
print("interface_example_backward: ", interface_example_backward)


Processing dataset: example
Dataset type: <class 'interface_experanto_temporaldata.InterfaceExperiment.InterfaceExperiment'>
Exp -> Td
interface_example_backward:  Data(
cursor=IrregularTimeSeries(
  timestamps=[66321],
  acc=[66321, 2],
  pos=[66321, 2],
  test_mask=[66321],
  train_mask=[66321],
  valid_mask=[66321],
  vel=[66321, 2]
),
cursor_outlier_segments=Interval(
  start=[7],
  end=[7],
  test_mask=[7],
  train_mask=[7],
  valid_mask=[7]
),
movement_phases=Data(
hold_period=Interval(
  start=[157],
  end=[157],
  test_mask=[157],
  train_mask=[157],
  valid_mask=[157]
),
invalid=Interval(
  start=[39],
  end=[39],
  test_mask=[39],
  train_mask=[39],
  valid_mask=[39]
),
random_period=Interval(
  start=[3],
  end=[3],
  test_mask=[3],
  train_mask=[3],
  valid_mask=[3]
),
reach_period=Interval(
  start=[157],
  end=[157],
  test_mask=[157],
  train_mask=[157],
  valid_mask=[157]
),
return_period=Interval(
  start=[156],
  end=[156],
  test_mask=[156],
  train_mask=[156],
  val

In [6]:
print("Checking equality of original and backward-converted Temporaldata objects...")
if td_check_equality(td_dataset, interface_example_backward):
    print("Success: The original and backward-converted Temporaldata objects are equal.")
else:    print("Failure: The original and backward-converted Temporaldata objects are NOT equal.")

Checking equality of original and backward-converted Temporaldata objects...
Success: The original and backward-converted Temporaldata objects are equal.


# Experanto -> Temporaldata -> Experanto

In [7]:
from experanto.interpolators import SequenceInterpolator, ScreenInterpolator, TimeIntervalInterpolator, SpikeInterpolator
def exp_check_equality(first, second):
    for dev in first.devices.keys():
        if dev in second.devices.keys():
            if isinstance(dev, SequenceInterpolator):
                if not (first.devices[dev].sampling_rate == second.devices[dev].sampling_rate and
                        first.devices[dev].start_time == second.devices[dev].start_time and
                        first.devices[dev].end_time == second.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval and
                        first.devices[dev].n_signals == second.devices[dev].n_signals):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
            if isinstance(dev, ScreenInterpolator):
                if not ((first.devices[dev].timestamps == second.devices[dev].timestamps).all() and
                        first.devices[dev].start_time == second.devices[dev].start_time and
                        first.devices[dev].end_time == second.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval and
                        first.devices[dev]._num_frames == second.devices[dev]._num_frames):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
            if isinstance(dev, TimeIntervalInterpolator):
                if not (first.devices[dev].start_time == second.devices[dev].start_time and
                        first.devices[dev].end_time == second.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
                if hasattr(first.devices[dev], "labeled_intervals") and hasattr(second.devices[dev], "labeled_intervals"):
                    if not first.devices[dev].labeled_intervals == second.devices[dev].labeled_intervals:
                        print(f"Metadata mismatch for device '{dev}'")
                        return False
                elif hasattr(first.devices[dev], "labeled_intervals") ^ hasattr(second.devices[dev], "labeled_intervals"):
                    print("No labeled_intervals attribute in one of the TimeIntervalInterpolators")
                    return False
            if isinstance(dev, SpikeInterpolator):
                if not (first.devices[dev].start_time == first.devices[dev].start_time and
                        first.devices[dev].end_time == first.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval and
                        first.devices[dev].n_signals == second.devices[dev].n_signals and
                        first.devices[dev].indices == second.devices[dev].indices and
                        (first.devices[dev].spikes == second.devices[dev].spikes).all):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
        else:
            print(f"Device '{dev}' not found in second object.")
            return False
    return True

### first example Experanto -> Temporaldata -> Experanto

In [8]:
from experanto.experiment import Experiment
root_folder = './dynamic29228-2-10-Video-full'
e = Experiment(root_folder)

/opt/anaconda3/envs/BA/lib/python3.11/site-packages/experanto/experiment.py:74: UserWarning: Falling back to original Interpolator creation logic.
  self._load_devices()


In [9]:
interface_dict = Interface({"example": e})
interface_example = interface_dict.process_as_temporaldata("example")

Processing dataset: example
Dataset type: <class 'experanto.experiment.Experiment'>
Exp -> Td


In [10]:
print("interface_example: ", interface_example)

interface_example:  Data(
eye_tracker=RegularTimeSeries(
  data=[150401, 4]
),
responses=RegularTimeSeries(
  data=[59912, 7928]
),
units=ArrayDict(
  id=[7928]
),
treadmill=RegularTimeSeries(
  data=[751971, 1]
),
screen=Data(
trial_0=IrregularTimeSeries(
  timestamps=[300],
  data=[300, 144, 256]
),
trial_1=IrregularTimeSeries(
  timestamps=[300],
  data=[300, 144, 256]
),
trial_2=IrregularTimeSeries(
  timestamps=[300],
  data=[300, 288, 512]
),
trial_3=IrregularTimeSeries(
  timestamps=[300],
  data=[300, 144, 256]
),
trial_4=IrregularTimeSeries(
  timestamps=[1],
  data=[1, 36, 64]
),
trial_5=IrregularTimeSeries(
  timestamps=[1],
  data=[1, 144, 256]
),
trial_6=IrregularTimeSeries(
  timestamps=[1],
  data=[1, 36, 64]
),
trial_7=IrregularTimeSeries(
  timestamps=[1],
  data=[1, 36, 64]
),
trial_8=IrregularTimeSeries(
  timestamps=[1],
  data=[1, 144, 256]
),
trial_9=IrregularTimeSeries(
  timestamps=[1],
  data=[1, 36, 64]
),
trial_10=IrregularTimeSeries(
  timestamps=[1],
  data

In [11]:
interface_dict_backward = Interface({"example": interface_example})
interface_dict_backward.custom_fns = {(td.Data, "screen"): screen_fn}
interface_example_backward = interface_dict_backward.process_as_experanto("example")
print("interface_example_backward: ", interface_example_backward)


Td -> Exp
interface_example_backward:  <interface_experanto_temporaldata.InterfaceExperiment.InterfaceExperiment object at 0x145a1f290>


In [12]:
print(interface_example_backward.devices.keys())
print(e.devices.keys())

dict_keys(['eye_tracker', 'responses', 'treadmill', 'screen'])
dict_keys(['eye_tracker', 'responses', 'treadmill', 'screen'])


In [13]:
print("Checking equality of original and backward-converted Experanto objects...")
if exp_check_equality(e, interface_example_backward):
    print("Success: The original and backward-converted Experanto objects are equal.")
else:    print("Failure: The original and backward-converted Experanto objects are NOT equal.")

Checking equality of original and backward-converted Experanto objects...
Success: The original and backward-converted Experanto objects are equal.


### second example Experanto -> Temporaldata -> Experanto

In [14]:
from experanto.configs import DEFAULT_MODALITY_CONFIG

config = dict(DEFAULT_MODALITY_CONFIG)
config["tiers"] = {
    "interpolation": {
        "_target_": "experanto.interpolators.TimeIntervalInterpolator"
    }
}
config["calcium"] = {
    "interpolation": {
        "_target_": "experanto.interpolators.SequenceInterpolator"
    }
}
e = Experiment(root_folder="./experanto_000039", modality_config=config)


In [15]:
interface_dict = Interface({"example": e})
interface_example = interface_dict.process_as_temporaldata("example")

Processing dataset: example
Dataset type: <class 'experanto.experiment.Experiment'>
Exp -> Td


In [16]:
print("interface_example: ", interface_example)

interface_example:  Data(
tiers=Interval(
  start=[1152],
  end=[1152],
  value=[1152]
),
calcium=RegularTimeSeries(
  data=[107115, 3]
),
)


In [17]:
interface_dict_backward = Interface({"example": interface_example})
interface_example_backward = interface_dict_backward.process_as_experanto("example")
print("interface_example_backward: ", interface_example_backward)

Td -> Exp
interface_example_backward:  <interface_experanto_temporaldata.InterfaceExperiment.InterfaceExperiment object at 0x1494b57d0>


In [18]:
print("Checking equality of original and backward-converted Experanto objects...")
if exp_check_equality(e, interface_example_backward):
    print("Success: The original and backward-converted Experanto objects are equal.")
else:    print("Failure: The original and backward-converted Experanto objects are NOT equal.")

Checking equality of original and backward-converted Experanto objects...
Success: The original and backward-converted Experanto objects are equal.
